# LeetCode 1914 — Cyclically Rotating a Grid bằng Python

Link đề: https://leetcode.com/problems/cyclically-rotating-a-grid/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Simulation

Skeleton LeetCode:

```python
class Solution:
    def rotateGrid(self, grid: List[List[int]], k: int) -> List[List[int]]:
        ...
```

Lộ trình: **mô phỏng từng bước xoay `k` lần** (chỉ chạy được với `k` nhỏ) → **mô phỏng với `k mod L`** → **tách từng lớp thành list, xoay bằng cắt lát, ghi lại** `O(m·n)` (lời giải nộp) → **ánh xạ chỉ số trực tiếp** (không cần list trung gian ngoài bản sao) → **`deque.rotate`**.

## 1. Mục tiêu học

1. Duyệt **viền (lớp) của ma trận** theo một chiều cố định (theo chiều kim đồng hồ) bằng bốn đoạn: cạnh trên, cạnh phải, cạnh dưới, cạnh trái — **không đếm trùng góc**.
2. Đổi "xoay ngược chiều kim đồng hồ 1 bước" thành "**dịch trái** list các phần tử của lớp (liệt kê theo chiều kim đồng hồ) đi 1 vị trí".
3. Dùng `k mod L` (với `L` là chu vi **của từng lớp**) để xử lý `k` tới `10⁹`.
4. Tránh lỗi **ghi đè khi đang đọc** bằng cách đọc ra list trước rồi ghi lại.
5. Kiểm chứng bằng mô phỏng từng bước (với `k` nhỏ) trên ma trận ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `grid` kích thước `m × n` (**`m` và `n` đều chẵn**) và số nguyên `k`. Ma trận gồm nhiều **lớp** lồng nhau (lớp ngoài cùng là viền ngoài, lớp tiếp theo là viền của ma trận con bên trong, …).

Một **lần xoay vòng** ma trận là xoay vòng **từng lớp** một lần: mỗi phần tử của lớp chiếm chỗ của phần tử **kề nó theo chiều ngược kim đồng hồ**.

Trả về ma trận sau `k` lần xoay.

**Ví dụ 1:** `grid = [[40,10],[30,20]], k = 1` → `[[10,20],[40,30]]`.

**Ví dụ 2:** `grid = [[1,2,3,4],[5,6,7,8],[9,10,11,12],[13,14,15,16]], k = 2` → `[[3,4,8,12],[2,11,10,16],[1,7,6,15],[5,9,13,14]]`.

**Ràng buộc:** `2 ≤ m, n ≤ 50`, `m`, `n` chẵn, `1 ≤ grid[i][j] ≤ 5000`, `1 ≤ k ≤ 10⁹`.

In [1]:
import copy
import random
import time
from collections import deque
from typing import List

examples = [
    ([[40, 10], [30, 20]], 1, [[10, 20], [40, 30]]),
    ([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]], 2,
     [[3, 4, 8, 12], [2, 11, 10, 16], [1, 7, 6, 15], [5, 9, 13, 14]]),
]

def show(g):
    w = max(len(str(x)) for row in g for x in row)
    for row in g:
        print("  " + " ".join(str(x).rjust(w) for x in row))

for g, k, want in examples:
    print("k =", k); show(g); print("->"); show(want); print()

k = 1
  40 10
  30 20
->
  10 20
  40 30

k = 2
   1  2  3  4
   5  6  7  8
   9 10 11 12
  13 14 15 16
->
   3  4  8 12
   2 11 10 16
   1  7  6 15
   5  9 13 14



## 3. Kiến thức nền

### 3.1 Lớp thứ `d` và chu vi của nó

Lớp `d` (tính từ 0) là viền của ma trận con hàng `d..m−1−d`, cột `d..n−1−d`. Có `min(m, n) / 2` lớp (vì `m`, `n` chẵn nên không có "lõi" một hàng/cột). Với chiều cao `h = m − 2d` và rộng `w = n − 2d`, số phần tử của lớp là `L = 2(h + w) − 4` (trừ 4 góc bị đếm hai lần).

In [2]:
m, n = 6, 8
for d in range(min(m, n) // 2):
    h, w = m - 2 * d, n - 2 * d
    print(f"lớp {d}: hàng {d}..{m-1-d}, cột {d}..{n-1-d}, h={h}, w={w}, L = 2(h+w)-4 = {2*(h+w)-4}")

lớp 0: hàng 0..5, cột 0..7, h=6, w=8, L = 2(h+w)-4 = 24
lớp 1: hàng 1..4, cột 1..6, h=4, w=6, L = 2(h+w)-4 = 16
lớp 2: hàng 2..3, cột 2..5, h=2, w=4, L = 2(h+w)-4 = 8


### 3.2 Liệt kê tọa độ một lớp theo chiều kim đồng hồ

Bốn đoạn, mỗi đoạn **không gồm** điểm cuối (điểm cuối là điểm đầu của đoạn sau): trên (trái → phải), phải (trên → dưới), dưới (phải → trái), trái (dưới → trên).

In [3]:
def layer_coords(m, n, d):
    top, bottom, left, right = d, m - 1 - d, d, n - 1 - d
    coords = []
    for j in range(left, right):          coords.append((top, j))       # cạnh trên
    for i in range(top, bottom):          coords.append((i, right))     # cạnh phải
    for j in range(right, left, -1):      coords.append((bottom, j))    # cạnh dưới
    for i in range(bottom, top, -1):      coords.append((i, left))      # cạnh trái
    return coords

print(layer_coords(4, 4, 0))
print(layer_coords(4, 4, 1))
print("số tọa độ lớp 0 của 6x8:", len(layer_coords(6, 8, 0)), "| không trùng:", len(set(layer_coords(6, 8, 0))) == 28)

[(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (3, 3), (3, 2), (3, 1), (3, 0), (2, 0), (1, 0)]
[(1, 1), (1, 2), (2, 2), (2, 1)]
số tọa độ lớp 0 của 6x8: 24 | không trùng: False


### 3.3 Xoay list bằng cắt lát

`a[k:] + a[:k]` dịch **trái** `k` vị trí (phần tử ở chỉ số `k` lên đầu). `a[-k:] + a[:-k]` dịch **phải**.

In [4]:
a = [1, 2, 3, 4, 5]
print("trái 2:", a[2:] + a[:2], "| phải 2:", a[-2:] + a[:-2])

trái 2: [3, 4, 5, 1, 2] | phải 2: [4, 5, 1, 2, 3]


### 3.4 `k mod L`

Xoay `L` lần là về nguyên trạng ⇒ chỉ cần `k % L`. Với `k = 10⁹` mà `L` nhỏ, điều này giảm khối lượng từ tỉ bước xuống vài bước.

In [5]:
L, k = 12, 10**9
print(k % L, "| kiểm: xoay", L, "lần một list:", (lambda a: a[L % len(a):] + a[:L % len(a)])(list(range(L))) == list(range(L)))

4 | kiểm: xoay 12 lần một list: True


### 3.5 Sao chép ma trận

`[row[:] for row in grid]` sao chép **sâu một tầng** — đủ cho ma trận số. `grid[:]` chỉ sao chép list ngoài (các hàng vẫn dùng chung!).

In [6]:
g = [[1, 2], [3, 4]]
shallow = g[:]; deep = [row[:] for row in g]
shallow[0][0] = 99
print("g bị ảnh hưởng qua bản sao nông:", g, "| bản sao từng hàng:", deep)

g bị ảnh hưởng qua bản sao nông: [[99, 2], [3, 4]] | bản sao từng hàng: [[1, 2], [3, 4]]


### 3.6 `deque.rotate`

`dq.rotate(-k)` dịch trái `k` — tương đương cắt lát, chạy `O(k)`.

In [7]:
dq = deque([1, 2, 3, 4, 5]); dq.rotate(-2); print(list(dq))

[3, 4, 5, 1, 2]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: lớp duy nhất, liệt kê theo chiều kim đồng hồ từ góc trên-trái: `[40, 10, 20, 30]` (trên-trái, trên-phải, dưới-phải, dưới-trái).

Sau 1 lần xoay **ngược** chiều kim đồng hồ, mỗi phần tử dời sang ô kề ngược chiều kim: `10` (trên-phải) sang trên-trái, `20` (dưới-phải) sang trên-phải, … ⇒ đọc lại theo chiều kim: `[10, 20, 30, 40]` — đúng là list cũ **dịch trái 1**.

```
40 10        10 20
30 20   ->   40 30
```

Ví dụ 2, lớp ngoài (12 phần tử) theo chiều kim: `1 2 3 4 8 12 16 15 14 13 9 5`; dịch trái 2 ⇒ `3 4 8 12 16 15 14 13 9 5 1 2` — ghi lại đúng thứ tự tọa độ ⇒ hàng đầu `3 4 8 12` ✔. Lớp trong `6 7 11 10` dịch trái 2 ⇒ `11 10 6 7` ⇒ hàng 1 giữa `11 10`, hàng 2 giữa `7 6` ✔ (đọc theo tọa độ `(1,1),(1,2),(2,2),(2,1)`).

In [8]:
g = examples[1][0]
for d in range(2):
    vals = [g[i][j] for i, j in layer_coords(4, 4, d)]
    print(f"lớp {d}: {vals} -> dịch trái 2 -> {vals[2:] + vals[:2]}")

lớp 0: [1, 2, 3, 4, 8, 12, 16, 15, 14, 13, 9, 5] -> dịch trái 2 -> [3, 4, 8, 12, 16, 15, 14, 13, 9, 5, 1, 2]
lớp 1: [6, 7, 11, 10] -> dịch trái 2 -> [11, 10, 6, 7]


## 5. Cách 1 — Brute Force / baseline: mô phỏng **từng bước** xoay, `k` lần

**Ý tưởng:** viết hàm xoay mọi lớp **một** bước (dịch trái 1 theo danh sách tọa độ), gọi `k` lần. Đúng theo câu chữ ⇒ dùng làm chuẩn đối chiếu (với `k` nhỏ).

In [9]:
def rotate_once(grid):
    m, n = len(grid), len(grid[0])
    out = [row[:] for row in grid]
    for d in range(min(m, n) // 2):
        cs = layer_coords(m, n, d)
        L = len(cs)
        for p, (i, j) in enumerate(cs):
            ni, nj = cs[(p + 1) % L]
            out[i][j] = grid[ni][nj]          # ô p nhận phần tử đang ở ô p+1 (kề theo chiều kim)
    return out

def rotate_brute(grid, k):
    for _ in range(k):
        grid = rotate_once(grid)
    return grid

for g, k, want in examples:
    print(rotate_brute(g, k) == want)

True
True


**Độ phức tạp:** mỗi bước `O(m·n)`, `k` bước ⇒ `O(k·m·n)`. Với `k = 10⁹` và `50 × 50` là `2.5·10¹²` — bất khả.

**Nút thắt:** xoay `L` lần thì lớp về nguyên trạng — phần lớn `k` bước là lãng phí.

In [10]:
g50 = [[random.Random(0).randint(1, 5000) for _ in range(50)] for _ in range(50)]
t = time.perf_counter(); rotate_brute(g50, 20); dt = time.perf_counter() - t
print(f"20 bước trên 50x50: {dt*1000:.0f} ms -> k = 10^9 ước tính ≈ {dt / 20 * 1e9 / 3600 / 24:.0f} ngày")

20 bước trên 50x50: 4 ms -> k = 10^9 ước tính ≈ 2 ngày


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — mỗi lớp độc lập:** xoay ma trận = xoay từng lớp; lớp `d` có chu vi `L_d`. Xoay `k` lần lớp `d` = xoay `k mod L_d` lần.

**Insight 2 — xoay là dịch list:** liệt kê tọa độ lớp theo chiều kim đồng hồ `c₀, c₁, …, c_{L−1}`. Sau một lần xoay ngược chiều kim, ô `c_p` nhận giá trị đang ở `c_{p+1}`. Sau `k` lần: ô `c_p` nhận giá trị ban đầu ở `c_{(p + k) mod L}`.

**Quy tắc:** với mỗi lớp: `vals = [grid[c] for c in coords]`, `r = k % L`, `vals = vals[r:] + vals[:r]`, ghi `vals[p]` vào `coords[p]`.

**Invariant:** các lớp rời nhau và phủ kín ma trận (vì `m`, `n` chẵn), nên xử lý từng lớp không ảnh hưởng lớp khác.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng từng bước nhưng với `k mod L` cho từng lớp

In [11]:
def rotate_mod_steps(grid, k):
    m, n = len(grid), len(grid[0])
    out = [row[:] for row in grid]
    for d in range(min(m, n) // 2):
        cs = layer_coords(m, n, d)
        L = len(cs)
        for _ in range(k % L):                      # tối đa L-1 bước
            first = out[cs[0][0]][cs[0][1]]
            for p in range(L - 1):                  # dời từng phần tử lùi 1 ô
                out[cs[p][0]][cs[p][1]] = out[cs[p + 1][0]][cs[p + 1][1]]
            out[cs[-1][0]][cs[-1][1]] = first
    return out

for g, k, want in examples:
    print(rotate_mod_steps(g, k) == want)

True
True


**Độ phức tạp:** lớp `d` tốn `O(L_d²)` worst (`k mod L_d` có thể tới `L_d − 1`) ⇒ tổng `O(Σ L_d²)` ≈ `O((m+n)² · min(m,n))` ≈ `200² · 25 = 10⁶` — chạy được, nhưng vẫn dư thừa (dời từng bước).

### 7.2 Cách 3 — Tách lớp, xoay bằng cắt lát, ghi lại (`O(m·n)`) — lời giải chuẩn

In [12]:
def rotate_slices_demo(grid, k):
    m, n = len(grid), len(grid[0])
    out = [row[:] for row in grid]
    for d in range(min(m, n) // 2):
        cs = layer_coords(m, n, d)
        vals = [grid[i][j] for i, j in cs]
        r = k % len(vals)
        vals = vals[r:] + vals[:r]
        print(f"lớp {d}: L={len(cs)}, k mod L = {r}, sau xoay {vals}")
        for (i, j), v in zip(cs, vals):
            out[i][j] = v
    return out

show(rotate_slices_demo(*examples[1][:2]))

lớp 0: L=12, k mod L = 2, sau xoay [3, 4, 8, 12, 16, 15, 14, 13, 9, 5, 1, 2]
lớp 1: L=4, k mod L = 2, sau xoay [11, 10, 6, 7]
   3  4  8 12
   2 11 10 16
   1  7  6 15
   5  9 13 14


In [13]:
class SolutionSlices:
    def rotateGrid(self, grid: List[List[int]], k: int) -> List[List[int]]:
        m, n = len(grid), len(grid[0])
        out = [row[:] for row in grid]
        for d in range(min(m, n) // 2):
            cs = layer_coords(m, n, d)
            vals = [grid[i][j] for i, j in cs]
            r = k % len(vals)
            vals = vals[r:] + vals[:r]
            for (i, j), v in zip(cs, vals):
                out[i][j] = v
        return out

for g, k, want in examples:
    print(SolutionSlices().rotateGrid(g, k) == want)

True
True


**Độ phức tạp:** mỗi ô thuộc đúng một lớp, được đọc và ghi một lần ⇒ `O(m·n)` thời gian, `O(m·n)` bộ nhớ (bản sao; có thể ghi đè trực tiếp vào `grid` vì đã đọc `vals` trước — `O(L)` bộ nhớ phụ).

### 7.3 Cách 4 — Ánh xạ chỉ số trực tiếp (không cắt lát)

Ô `c_p` nhận giá trị gốc ở `c_{(p+k) mod L}` — đọc từ **bản gốc**, ghi vào **bản mới**.

In [14]:
class SolutionIndex:
    def rotateGrid(self, grid: List[List[int]], k: int) -> List[List[int]]:
        m, n = len(grid), len(grid[0])
        out = [row[:] for row in grid]
        for d in range(min(m, n) // 2):
            cs = layer_coords(m, n, d)
            L = len(cs); r = k % L
            for p, (i, j) in enumerate(cs):
                si, sj = cs[(p + r) % L]
                out[i][j] = grid[si][sj]
        return out

for g, k, want in examples:
    print(SolutionIndex().rotateGrid(g, k) == want)

True
True


### 7.4 Cách 5 — `deque.rotate` và ghi tại chỗ

In [15]:
class SolutionDeque:
    def rotateGrid(self, grid: List[List[int]], k: int) -> List[List[int]]:
        m, n = len(grid), len(grid[0])
        g = [row[:] for row in grid]                # không sửa input của người gọi
        for d in range(min(m, n) // 2):
            cs = layer_coords(m, n, d)
            dq = deque(g[i][j] for i, j in cs)     # đọc hết trước
            dq.rotate(-(k % len(cs)))
            for (i, j), v in zip(cs, dq):          # rồi mới ghi
                g[i][j] = v
        return g

for g, k, want in examples:
    print(SolutionDeque().rotateGrid(g, k) == want)

True
True


Cả hai cách đều `O(m·n)`. Cách 4 tránh tạo list xoay; cách 5 cho thấy "đọc hết rồi ghi" giúp ghi **tại chỗ** an toàn.

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 1 (`L = 4`, `k mod L = 1`), tọa độ theo chiều kim `c₀=(0,0), c₁=(0,1), c₂=(1,1), c₃=(1,0)`:

| p | ô `c_p` | nguồn `c_{(p+1) mod 4}` | giá trị gốc ở nguồn | ô mới |
|---|---|---|---|---|
| 0 | (0,0) | (0,1) | 10 | 10 |
| 1 | (0,1) | (1,1) | 20 | 20 |
| 2 | (1,1) | (1,0) | 30 | 30 |
| 3 | (1,0) | (0,0) | 40 | 40 |

Kết quả `[[10,20],[40,30]]`. ✔

In [16]:
def trace_index(grid, k):
    m, n = len(grid), len(grid[0])
    out = [row[:] for row in grid]
    for d in range(min(m, n) // 2):
        cs = layer_coords(m, n, d); L = len(cs); r = k % L
        print(f"lớp {d}: L = {L}, k mod L = {r}")
        for p, (i, j) in enumerate(cs):
            si, sj = cs[(p + r) % L]
            out[i][j] = grid[si][sj]
            print(f"  p={p:>2} ô {(i, j)} <- nguồn {(si, sj)} giá trị {grid[si][sj]}")
    return out

show(trace_index(*examples[0][:2]))

lớp 0: L = 4, k mod L = 1
  p= 0 ô (0, 0) <- nguồn (0, 1) giá trị 10
  p= 1 ô (0, 1) <- nguồn (1, 1) giá trị 20
  p= 2 ô (1, 1) <- nguồn (1, 0) giá trị 30
  p= 3 ô (1, 0) <- nguồn (0, 0) giá trị 40
  10 20
  40 30


In [17]:
res = trace_index([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]], 2)
show(res)

lớp 0: L = 12, k mod L = 2
  p= 0 ô (0, 0) <- nguồn (0, 2) giá trị 3
  p= 1 ô (0, 1) <- nguồn (0, 3) giá trị 4
  p= 2 ô (0, 2) <- nguồn (1, 3) giá trị 8
  p= 3 ô (0, 3) <- nguồn (2, 3) giá trị 12
  p= 4 ô (1, 3) <- nguồn (3, 3) giá trị 16
  p= 5 ô (2, 3) <- nguồn (3, 2) giá trị 15
  p= 6 ô (3, 3) <- nguồn (3, 1) giá trị 14
  p= 7 ô (3, 2) <- nguồn (3, 0) giá trị 13
  p= 8 ô (3, 1) <- nguồn (2, 0) giá trị 9
  p= 9 ô (3, 0) <- nguồn (1, 0) giá trị 5
  p=10 ô (2, 0) <- nguồn (0, 0) giá trị 1
  p=11 ô (1, 0) <- nguồn (0, 1) giá trị 2
lớp 1: L = 4, k mod L = 2
  p= 0 ô (1, 1) <- nguồn (2, 2) giá trị 11
  p= 1 ô (1, 2) <- nguồn (2, 1) giá trị 10
  p= 2 ô (2, 2) <- nguồn (1, 1) giá trị 6
  p= 3 ô (2, 1) <- nguồn (1, 2) giá trị 7
   3  4  8 12
   2 11 10 16
   1  7  6 15
   5  9 13 14


## 9. Code nộp LeetCode

In [18]:
from typing import List

class Solution:
    def rotateGrid(self, grid: List[List[int]], k: int) -> List[List[int]]:
        m, n = len(grid), len(grid[0])
        for d in range(min(m, n) // 2):
            top, bottom, left, right = d, m - 1 - d, d, n - 1 - d
            coords = ([(top, j) for j in range(left, right)] +
                      [(i, right) for i in range(top, bottom)] +
                      [(bottom, j) for j in range(right, left, -1)] +
                      [(i, left) for i in range(bottom, top, -1)])   # theo chiều kim đồng hồ
            vals = [grid[i][j] for i, j in coords]                    # đọc hết trước
            r = k % len(vals)
            vals = vals[r:] + vals[:r]                                # ngược chiều kim = dịch trái
            for (i, j), v in zip(coords, vals):
                grid[i][j] = v
        return grid

for g, k, want in examples:
    assert Solution().rotateGrid([row[:] for row in g], k) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


(Lời giải nộp ghi **tại chỗ** vào `grid` — an toàn vì mỗi lớp đã được đọc hết vào `vals` trước khi ghi; LeetCode chấp nhận việc sửa input. Khi test cục bộ ta truyền bản sao.)

## 10. Vì sao thuật toán đúng

**Các lớp phân hoạch ma trận.** Ô `(i, j)` thuộc lớp `d = min(i, j, m−1−i, n−1−j)`. Vì `m`, `n` chẵn, mọi lớp đều là một "viền hình chữ nhật" có chiều cao, chiều rộng ≥ 2, nên `layer_coords` liệt kê đúng `2(h+w) − 4` ô phân biệt và không có lõi lẻ.

**Một lần xoay = dịch trái 1.** Tọa độ `c₀ … c_{L−1}` liệt kê theo chiều kim đồng hồ và `c_{p+1}` kề `c_p` (theo chiều kim), `c₀` kề `c_{L−1}`. "Mỗi phần tử chiếm chỗ ô kề theo chiều **ngược** kim" ⇔ phần tử ở `c_{p+1}` chuyển sang `c_p` ⇔ giá trị mới tại `c_p` = giá trị cũ tại `c_{p+1}` ⇔ list giá trị dịch trái 1.

**`k` lần = dịch trái `k mod L`.** Quy nạp: sau `t` lần, `c_p` chứa giá trị gốc ở `c_{(p+t) mod L}`. Dịch trái là phép quay của nhóm cyclic cấp `L` ⇒ chỉ phụ thuộc `t mod L`.

**Ghi đúng:** đọc toàn bộ `vals` của lớp trước khi ghi ⇒ không dùng giá trị đã bị ghi đè; các lớp rời nhau ⇒ thứ tự xử lý lớp không quan trọng.

## 11. Phân tích Time / Space

Gọi `m, n ≤ 50`, số lớp `D = min(m, n)/2`, chu vi lớp `L_d`.

| Cách | Time | Space |
|---|---|---|
| 1. Từng bước, `k` lần | `O(k · m · n)` | `O(m · n)` |
| 2. Từng bước, `k mod L` | `O(Σ L_d²)` | `O(m · n)` |
| 3. Cắt lát | `O(m · n)` | `O(m · n)` hoặc `O(L)` phụ nếu ghi tại chỗ |
| 4. Ánh xạ chỉ số | `O(m · n)` | `O(m · n)` |
| 5. Deque | `O(m · n)` | `O(L)` phụ |

`k` tới `10⁹` không ảnh hưởng thời gian của cách 3–5 (chỉ dùng `k % L`).

In [19]:
rng = random.Random(1914)
g = [[rng.randint(1, 5000) for _ in range(50)] for _ in range(50)]
for name, f in [("Cách 2", rotate_mod_steps), ("Cách 3", SolutionSlices().rotateGrid),
                ("Cách 4", SolutionIndex().rotateGrid), ("Cách 5", SolutionDeque().rotateGrid),
                ("Nộp", lambda gg, kk: Solution().rotateGrid([r[:] for r in gg], kk))]:
    t = time.perf_counter()
    for k in (1, 10**9, 123456789, 97):
        f(g, k)
    print(f"{name}: 4 lần xoay 50x50 trong {1000*(time.perf_counter()-t):.1f} ms")

Cách 2: 4 lần xoay 50x50 trong 27.2 ms
Cách 3: 4 lần xoay 50x50 trong 0.8 ms
Cách 4: 4 lần xoay 50x50 trong 0.8 ms
Cách 5: 4 lần xoay 50x50 trong 0.9 ms
Nộp: 4 lần xoay 50x50 trong 0.9 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu mô phỏng từng bước)

In [20]:
def call(sol, g, k):
    return sol.rotateGrid([row[:] for row in g], k)

sols = [Solution(), SolutionSlices(), SolutionIndex(), SolutionDeque()]
g24 = [[1, 2, 3, 4], [5, 6, 7, 8]]                      # 2 x 4: một lớp, L = 8
g42 = [[1, 2], [3, 4], [5, 6], [7, 8]]                  # 4 x 2
edge = [(g, k) for g, k, _ in examples] + [
    (g24, 1), (g24, 8), (g24, 9), (g42, 3),
    ([[1, 2], [3, 4]], 4),                              # k = L: không đổi
    ([[1, 2], [3, 4]], 10**9),
    ([[i * 6 + j for j in range(6)] for i in range(4)], 7),
]
for g, k in edge:
    want = rotate_brute(g, k) if k <= 1000 else rotate_mod_steps(g, k)
    for s in sols:
        assert call(s, g, k) == want, (g, k, s)
    print("OK", f"{len(g)}x{len(g[0])}", "k =", k)

OK 2x2 k = 1
OK 4x4 k = 2
OK 2x4 k = 1
OK 2x4 k = 8
OK 2x4 k = 9
OK 4x2 k = 3
OK 2x2 k = 4
OK 2x2 k = 1000000000
OK 4x6 k = 7


(Với `k = 10⁹` ta đối chiếu bằng Cách 2 — mô phỏng từng bước nhưng `k mod L` — vì mô phỏng thuần `10⁹` bước là bất khả; Cách 2 tự nó đã được kiểm với mô phỏng thuần ở 12.2.)

### 12.2 Ngẫu nhiên có seed: so với mô phỏng từng bước thuần

In [21]:
rng = random.Random(42)
for _ in range(300):
    m, n = rng.choice([2, 4, 6, 8]), rng.choice([2, 4, 6, 8])
    g = [[rng.randint(1, 5000) for _ in range(n)] for _ in range(m)]
    k = rng.randint(1, 40)
    want = rotate_brute(g, k)
    assert rotate_mod_steps(g, k) == want
    for s in sols:
        assert call(s, g, k) == want
for _ in range(100):
    m, n = rng.choice(range(2, 51, 2)), rng.choice(range(2, 51, 2))
    g = [[rng.randint(1, 5000) for _ in range(n)] for _ in range(m)]
    k = rng.randint(1, 10**9)
    want = rotate_mod_steps(g, k)
    assert all(call(s, g, k) == want for s in sols)
print("300 case k nhỏ (mô phỏng thuần) + 100 case k tới 10^9 (Cách 2): khớp")

300 case k nhỏ (mô phỏng thuần) + 100 case k tới 10^9 (Cách 2): khớp


### 12.3 Tính chất: xoay `k₁` rồi `k₂` = xoay `k₁ + k₂`; xoay `lcm(L_d)` lần về nguyên trạng

In [22]:
from math import lcm
rng = random.Random(5)
for _ in range(100):
    m, n = rng.choice([2, 4, 6]), rng.choice([2, 4, 6, 8])
    g = [[rng.randint(1, 99) for _ in range(n)] for _ in range(m)]
    k1, k2 = rng.randint(1, 10**9), rng.randint(1, 10**9)
    assert call(Solution(), call(Solution(), g, k1), k2) == call(Solution(), g, k1 + k2)
    period = lcm(*[len(layer_coords(m, n, d)) for d in range(min(m, n) // 2)])
    assert call(Solution(), g, period) == g
print("tính chất cộng và chu kỳ đúng")

tính chất cộng và chu kỳ đúng


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không lấy `k mod L` (mô phỏng `k` bước)

In [23]:
g = examples[1][0]
t = time.perf_counter(); rotate_brute(g, 2000); dt = time.perf_counter() - t
print(f"2000 bước trên 4x4: {1000*dt:.0f} ms -> k = 10^9 ≈ {dt / 2000 * 1e9 / 60:.0f} phút (chỉ 4x4!)")
print("với k mod L:", call(Solution(), g, 10**9) == rotate_mod_steps(g, 10**9))

2000 bước trên 4x4: 5 ms -> k = 10^9 ≈ 38 phút (chỉ 4x4!)
với k mod L: True


**Vì sao sai:** đúng nhưng `k` tới `10⁹` ⇒ TLE. **Sửa:** mỗi lớp dùng `k % L`.

### Lỗi 2 — Xoay sai chiều (theo chiều kim đồng hồ)

In [24]:
def wrong_direction(grid, k):
    m, n = len(grid), len(grid[0]); out = [row[:] for row in grid]
    for d in range(min(m, n) // 2):
        cs = layer_coords(m, n, d); vals = [grid[i][j] for i, j in cs]; r = k % len(vals)
        vals = vals[-r:] + vals[:-r] if r else vals       # SAI: dịch phải = theo chiều kim
        for (i, j), v in zip(cs, vals): out[i][j] = v
    return out

g, k, want = examples[0]
print("sai:", wrong_direction(g, k), "| đúng:", want)

sai: [[30, 40], [20, 10]] | đúng: [[10, 20], [40, 30]]


**Vì sao sai:** "phần tử chiếm chỗ ô kề **ngược** chiều kim" ⇒ với danh sách liệt kê **theo** chiều kim, đó là dịch **trái**. Kiểm nhanh bằng Ví dụ 1 trước khi nộp.

### Lỗi 3 — Đếm trùng góc khi liệt kê lớp

In [25]:
def wrong_coords(m, n, d):
    top, bottom, left, right = d, m - 1 - d, d, n - 1 - d
    cs = [(top, j) for j in range(left, right + 1)]               # gồm cả góc phải
    cs += [(i, right) for i in range(top, bottom + 1)]            # lại gồm góc trên-phải
    cs += [(bottom, j) for j in range(right, left - 1, -1)]
    cs += [(i, left) for i in range(bottom, top - 1, -1)]
    return cs                                                      # SAI: 4 góc xuất hiện 2 lần

def rotate_with(coords_fn, grid, k):
    m, n = len(grid), len(grid[0]); out = [row[:] for row in grid]
    for d in range(min(m, n) // 2):
        cs = coords_fn(m, n, d); vals = [grid[i][j] for i, j in cs]; r = k % len(vals)
        vals = vals[r:] + vals[:r]
        for (i, j), v in zip(cs, vals): out[i][j] = v
    return out

g, k, want = examples[1]
print("L sai =", len(wrong_coords(4, 4, 0)), "(đúng là 12)")
print("sai:", rotate_with(wrong_coords, g, k) == want, "| đúng:", rotate_with(layer_coords, g, k) == want)

L sai = 16 (đúng là 12)
sai: False | đúng: True


**Vì sao sai:** mỗi góc nằm ở hai cạnh. Nếu cả hai đoạn đều gồm góc, chu vi bị tính `2(h+w)` thay vì `2(h+w) − 4`, và khi ghi lại, góc nhận giá trị hai lần (lần sau đè lần trước) ⇒ sai. **Sửa:** mỗi đoạn **bỏ điểm cuối** (nửa mở).

### Lỗi 4 — Dùng cùng một `k mod L` (của lớp ngoài) cho mọi lớp

In [26]:
def wrong_same_mod(grid, k):
    m, n = len(grid), len(grid[0]); out = [row[:] for row in grid]
    L0 = len(layer_coords(m, n, 0)); r = k % L0                  # SAI: lớp trong có L khác
    for d in range(min(m, n) // 2):
        cs = layer_coords(m, n, d); vals = [grid[i][j] for i, j in cs]
        rr = r % len(vals)                                       # = (k mod L0) mod L_d
        vals = vals[rr:] + vals[:rr]
        for (i, j), v in zip(cs, vals): out[i][j] = v
    return out

g = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]
for k in (13, 18):
    print("4x4, k =", k, "| bản sai khớp đáp án?", wrong_same_mod(g, k) == call(Solution(), g, k))

4x4, k = 13 | bản sai khớp đáp án? True
4x4, k = 18 | bản sai khớp đáp án? True


**Vì sao sai (và vì sao khó phát hiện):** bản sai tính `(k mod L₀) mod L_d` thay vì `k mod L_d`. Hai giá trị bằng nhau **khi `L_d` chia hết `L₀`** — như ma trận `4 × 4` (`L₀ = 12`, `L₁ = 4`), nên ở trên in `True` với mọi `k` và test theo ví dụ đề **không bắt được lỗi**. Với `6 × 8`: `L₀ = 24`, `L₁ = 16`, `L₂ = 8` — `16` không chia hết `24`. Chọn `k = 30`: lớp 1 cần xoay `30 mod 16 = 14`, bản sai xoay `(30 mod 24) mod 16 = 6`. Kiểm trên `6 × 8`:

In [27]:
g68 = [[i * 8 + j for j in range(8)] for i in range(6)]
for k in (17, 30):
    print("6x8, k =", k, "| bản sai khớp đáp án?", wrong_same_mod(g68, k) == call(Solution(), g68, k))

6x8, k = 17 | bản sai khớp đáp án? True
6x8, k = 30 | bản sai khớp đáp án? False


**Sửa:** tính `k % L` **riêng cho từng lớp**.

### Lỗi 5 — Ghi đè tại chỗ trong khi còn đang đọc

In [28]:
def wrong_inplace(grid, k):
    g = [row[:] for row in grid]; m, n = len(g), len(g[0])
    for d in range(min(m, n) // 2):
        cs = layer_coords(m, n, d); L = len(cs); r = k % L
        for p, (i, j) in enumerate(cs):
            si, sj = cs[(p + r) % L]
            g[i][j] = g[si][sj]                   # SAI: nguồn có thể đã bị ghi đè ở bước trước
    return g

g, k, want = examples[1]
print("sai:", wrong_inplace(g, k) == want); show(wrong_inplace(g, k))

sai: False
   3  4  8 12
   4 11 10 16
   3 10 11 15
   5  9 13 14


**Vì sao sai:** khi `p` lớn, nguồn `c_{(p+r) mod L}` vòng về đầu danh sách — các ô đầu **đã bị ghi** giá trị mới. **Sửa:** đọc từ bản gốc (Cách 4) hoặc đọc hết `vals` trước (Cách 3, 5).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Xoay/duyệt viền ma trận theo lớp"** (LeetCode 54 Spiral Matrix, 48 Rotate Image, 1914) ⇒ viết **một** hàm sinh tọa độ lớp bằng 4 đoạn **nửa mở**.
- **Xoay vòng `k` lần** ⇒ `k mod chu_kỳ`; chu kỳ có thể **khác nhau** giữa các thành phần.
- "Đọc hết rồi ghi" để tránh ghi đè.
- Xác định chiều bằng ví dụ nhỏ nhất (2 × 2) trước khi viết code.
- Mỗi ô thuộc lớp `min(i, j, m−1−i, n−1−j)`.

## 15. Biến thể của bài

1. **Xoay theo chiều kim đồng hồ** ⇒ dịch phải (Bài tập 4).
2. **Mỗi lớp xoay chiều khác nhau / số bước khác nhau** (Bài tập 5).
3. **`m`, `n` lẻ** ⇒ có "lõi" là một hàng/cột/ô đơn — lõi một hàng/cột cũng là một vòng (đi ra rồi đi về) cần định nghĩa riêng; ô đơn đứng yên.
4. **LeetCode 48 — Rotate Image** (xoay cả ma trận 90°): mỗi lớp dịch `w − 1` vị trí — chính là trường hợp riêng của bài này với ma trận vuông!
5. **LeetCode 54/59 — Spiral Matrix I/II** — cùng kỹ thuật duyệt lớp.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từng bước `k` lần | đúng câu chữ | `O(k·m·n)` | `O(mn)` | chuẩn đối chiếu, `k` nhỏ |
| 2. Từng bước `k mod L` | giảm `k` | `O(Σ L²)` | `O(mn)` | khi chỉ có hàm "xoay 1 bước" |
| 3. Cắt lát | `vals[r:] + vals[:r]` | `O(mn)` | `O(L)` phụ | **nộp bài** |
| 4. Ánh xạ chỉ số | `c_p ← c_{(p+r) mod L}` | `O(mn)` | `O(mn)` | không cần list trung gian |
| 5. Deque | `rotate(−r)` | `O(mn)` | `O(L)` | quen API deque |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `rotateGrid([[1,2],[3,4]], 4)` và `rotateGrid([[1,2],[3,4]], 2)`?

**Bài 2 (tính tay):** Với `grid` `2 × 4` = `[[1,2,3,4],[5,6,7,8]]`, `k = 3`: viết list lớp theo chiều kim, dịch, ghi lại.

**Bài 3 (điền TODO):** Viết `layer_of(m, n, i, j)` trả về chỉ số lớp của ô `(i, j)` và `layer_count(m, n)`.

**Bài 4 (tự viết lại):** Viết `rotate_clockwise(grid, k)` — xoay **theo** chiều kim đồng hồ `k` lần.

**Bài 5 (mini challenge):** Viết `rotate_alternating(grid, k)`: lớp `d` chẵn xoay **ngược** chiều kim `k` lần, lớp lẻ xoay **theo** chiều kim `k` lần. Đối chiếu bằng mô phỏng từng bước.

In [29]:
def layer_of(m, n, i, j):
    # TODO
    pass

def rotate_clockwise(grid, k):
    # TODO
    pass

def rotate_alternating(grid, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Lớp duy nhất, `L = 4` | `4 mod 4 = 0` ⇒ không đổi | `k = 2`: `[[4,3],[2,1]]` |
| 2 | List: `1 2 3 4 8 7 6 5` | Dịch trái 3: `4 8 7 6 5 1 2 3` | Hàng trên `4 8 7 6`, hàng dưới (đọc ngược) … |
| 3 | `min(i, j, m−1−i, n−1−j)` | Số lớp `min(m, n) // 2` | Kiểm bằng `layer_coords` |
| 4 | Theo chiều kim = dịch phải | `vals[-r:] + vals[:-r]` (cẩn thận `r = 0`) | Hoặc `vals[(L − r) % L:] + …` |
| 5 | Chọn hướng theo `d % 2` | Hàm xoay 1 bước có tham số hướng | Mô phỏng: gọi hàm 1 bước `k` lần |

## 19. Đáp án tham khảo

In [30]:
print("Bài 1:", call(Solution(), [[1, 2], [3, 4]], 4), call(Solution(), [[1, 2], [3, 4]], 2))
assert call(Solution(), [[1, 2], [3, 4]], 4) == [[1, 2], [3, 4]]
assert call(Solution(), [[1, 2], [3, 4]], 2) == [[4, 3], [2, 1]]
print("Bài 2:"); r2 = trace_index([[1, 2, 3, 4], [5, 6, 7, 8]], 3); show(r2)
assert r2 == [[4, 8, 7, 6], [3, 2, 1, 5]] == rotate_brute([[1, 2, 3, 4], [5, 6, 7, 8]], 3)

Bài 1: [[1, 2], [3, 4]] [[4, 3], [2, 1]]
Bài 2:
lớp 0: L = 8, k mod L = 3
  p= 0 ô (0, 0) <- nguồn (0, 3) giá trị 4
  p= 1 ô (0, 1) <- nguồn (1, 3) giá trị 8
  p= 2 ô (0, 2) <- nguồn (1, 2) giá trị 7
  p= 3 ô (0, 3) <- nguồn (1, 1) giá trị 6
  p= 4 ô (1, 3) <- nguồn (1, 0) giá trị 5
  p= 5 ô (1, 2) <- nguồn (0, 0) giá trị 1
  p= 6 ô (1, 1) <- nguồn (0, 1) giá trị 2
  p= 7 ô (1, 0) <- nguồn (0, 2) giá trị 3
  4 8 7 6
  3 2 1 5


In [31]:
def layer_of(m, n, i, j):
    return min(i, j, m - 1 - i, n - 1 - j)

def layer_count(m, n):
    return min(m, n) // 2

def rotate_clockwise(grid, k):
    m, n = len(grid), len(grid[0]); out = [row[:] for row in grid]
    for d in range(layer_count(m, n)):
        cs = layer_coords(m, n, d); L = len(cs); r = k % L
        vals = [grid[i][j] for i, j in cs]
        vals = vals[(L - r) % L:] + vals[:(L - r) % L]
        for (i, j), v in zip(cs, vals): out[i][j] = v
    return out

rng = random.Random(3)
for _ in range(200):
    m, n = rng.choice([2, 4, 6]), rng.choice([2, 4, 6, 8])
    for d in range(layer_count(m, n)):
        assert all(layer_of(m, n, i, j) == d for i, j in layer_coords(m, n, d))
    assert sum(len(layer_coords(m, n, d)) for d in range(layer_count(m, n))) == m * n
    g = [[rng.randint(1, 99) for _ in range(n)] for _ in range(m)]
    k = rng.randint(0, 30)
    # xoay theo chiều kim k lần rồi ngược chiều k lần phải về như cũ
    assert call(Solution(), rotate_clockwise(g, k), k) == g if k else True
print("Bài 3, 4: OK", layer_of(6, 8, 2, 5))

Bài 3, 4: OK 2


In [32]:
def step_layer(out, grid, cs, direction):
    L = len(cs)
    for p, (i, j) in enumerate(cs):
        si, sj = cs[(p + direction) % L]
        out[i][j] = grid[si][sj]

def rotate_alternating(grid, k):
    m, n = len(grid), len(grid[0]); out = [row[:] for row in grid]
    for d in range(layer_count(m, n)):
        cs = layer_coords(m, n, d); L = len(cs)
        r = k % L if d % 2 == 0 else (-k) % L          # lớp lẻ: theo chiều kim = dịch phải
        vals = [grid[i][j] for i, j in cs]
        vals = vals[r:] + vals[:r]
        for (i, j), v in zip(cs, vals): out[i][j] = v
    return out

def rotate_alternating_sim(grid, k):
    m, n = len(grid), len(grid[0])
    for _ in range(k):
        out = [row[:] for row in grid]
        for d in range(layer_count(m, n)):
            step_layer(out, grid, layer_coords(m, n, d), 1 if d % 2 == 0 else -1)
        grid = out
    return grid

rng = random.Random(9)
for _ in range(200):
    m, n = rng.choice([2, 4, 6, 8]), rng.choice([4, 6, 8])
    g = [[rng.randint(1, 99) for _ in range(n)] for _ in range(m)]
    k = rng.randint(1, 30)
    assert rotate_alternating(g, k) == rotate_alternating_sim(g, k)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK")
show(rotate_alternating(examples[1][0], 1))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK
   2  3  4  8
   1 10  6 12
   5 11  7 16
   9 13 14 15


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
for d in range(min(m, n) // 2):
    t, b, l, r = d, m - 1 - d, d, n - 1 - d
    cs = ([(t, j) for j in range(l, r)] + [(i, r) for i in range(t, b)] +
          [(b, j) for j in range(r, l, -1)] + [(i, l) for i in range(b, t, -1)])  # chiều kim, nửa mở
    vals = [grid[i][j] for i, j in cs]          # đọc hết trước
    s = k % len(vals)                            # k mod L của TỪNG lớp
    vals = vals[s:] + vals[:s]                   # ngược chiều kim = dịch trái
    for (i, j), v in zip(cs, vals): grid[i][j] = v
```

**Checklist:**
- [ ] 4 đoạn nửa mở, chu vi `2(h + w) − 4` (không đếm trùng góc)?
- [ ] Ngược chiều kim đồng hồ = **dịch trái** list liệt kê theo chiều kim (kiểm bằng Ví dụ 1)?
- [ ] `k % L` **riêng từng lớp**?
- [ ] Đọc hết giá trị lớp trước khi ghi?
- [ ] Số lớp `min(m, n) // 2` (m, n chẵn nên không có lõi).